In [27]:
import numpy as np
import matplotlib.pyplot as plt
import datasets

In [28]:
mnist_ds = datasets.load_from_disk("ylecun/mnist").with_format("numpy")

# 2. Extract contiguous blocks directly (instant PyArrow extraction)
X_train = (mnist_ds["train"]["image"][:] / 255.0).astype(np.float32)
Y_train = mnist_ds["train"]["label"][:].astype(np.int64)

X_test = (mnist_ds["test"]["image"][:] / 255.0).astype(np.float32)
Y_test = mnist_ds["test"]["label"][:].astype(np.int64)

# 3. Fast in-memory shuffle using NumPy
rng = np.random.default_rng(42)

train_idx = rng.permutation(len(X_train))
X_train, Y_train = X_train[train_idx], Y_train[train_idx]

test_idx = rng.permutation(len(X_test))
X_test, Y_test = X_test[test_idx], Y_test[test_idx]

# one hotting
Y_train = np.eye(10, dtype=np.float32)[Y_train]   
Y_test = np.eye(10, dtype=np.float32)[Y_test]     

# flatening and changing resolution
X_train = X_train.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4)).reshape(len(X_train), -1)  
X_test = X_test.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4)).reshape(len(X_test), -1)     

In [ ]:
from abc import ABC, abstractmethod
from typing import Iterator
from numpy.typing import NDArray

class Trainer():
    def __init__(self, optimizer: Optimizer, x_train_set: NDArray, y_train_set: NDArray , batching=False, chunk_size=-1, batch_size = -1):
        self.optimizer = optimizer
        self.batching = batching
        self.chunk_size = chunk_size
        self.batch_size = batch_size
        self.x_train_set = x_train_set
        self.y_train_set = y_train_set

    def train(self, n_steps=-1, report_freq=1):
        size = self.batch_size if self.batching else self.chunk_size
    
        for i in range(n_steps):
            s, e = i * size, (i + 1) * size
            x, y = self.x_train_set[s:e], self.y_train_set[s:e]
    
            if self.batching:
                loss = self.optimizer.step_accumulate_batch(x, y)
            else:
                loss = sum(self.optimizer.step_accumulate_gradients(xi, yi) for xi, yi in zip(x, y))
            self.optimizer.step_learn()
    
            loss /= size
            if i % report_freq == 0: print(f"step: {i} loss = {loss}")



class Model(ABC):

    layers: list

    @abstractmethod
    def accumulate_gradients(self, gradients_forward): pass
    
    def zero_grad(self):
        for layer in self.layers: layer.zero_grad()

    @abstractmethod
    def __call__(self, x): pass

    @abstractmethod
    def parameters(self) -> Iterator[tuple[NDArray[np.float32], NDArray[np.float32]]]: pass

    @abstractmethod
    def forward_batch(self, x): pass

    @abstractmethod
    def accumulate_gradients_batch(self, gradients_forward): pass

    
class Optimizer(ABC):
    
    @abstractmethod
    def step_accumulate_gradients(self, x, y) -> float: pass

    @abstractmethod
    def step_learn(self): pass

    @abstractmethod
    def step_accumulate_batch(self, x, y) -> float: pass



class Layer():
    def __init__(self, nin, nout, nonlin, seed=42): # eah nout is a neuron
        self._tmp = np.zeros((nout, nin), dtype=np.float32) # used as a buffer to avoid reallocating constantly
        self.nout = nout
        self.nin = nin
        rng = np.random.default_rng(seed)

        self.weights = rng.standard_normal((nout, nin), dtype=np.float32) * np.float32(np.sqrt(2.0 / nin)) 
        self.gradients = np.zeros((nout, nin), dtype=np.float32)

        self.bias_weights = rng.standard_normal((nout,), dtype=np.float32) * np.float32(np.sqrt(2.0 / nin)) 
        self.bias_gradients = np.zeros((nout,), dtype=np.float32)

        self.mask = np.ones((nout,), dtype=np.float32)
        self.nonlin = nonlin
        self.activationsin = np.zeros((nin,), dtype=np.float32)
        self.activationsout = np.zeros((nout,), dtype=np.float32)
        

    def __call__(self, activation): # single inference
        self.activationsin = activation
        neurons_out = (self.weights @ activation) + self.bias_weights
        if self.nonlin:
            self.mask = neurons_out > 0 # ReLu activation function
        self.activationsout = neurons_out * self.mask
        return self.activationsout
    

    def forward_batch(self, activation):  # (B, nin) -> (B, nout)
        self.activationsin = activation
        neurons_out = activation @ self.weights.T + self.bias_weights
        if self.nonlin:
            self.mask = neurons_out > 0  # ReLU
        self.activationsout = neurons_out * self.mask
        return self.activationsout


    def accumulate_gradients_batch(self, gradients_forward):  # (B, nout) -> (B, nin)
        g = gradients_forward * self.mask                    # kills dead neurons
        np.matmul(g.T, self.activationsin, out=self._tmp)    # (nout, nin), sum of per-sample outer products
        self.gradients += self._tmp
        self.bias_gradients += g.sum(axis=0)
        return g @ self.weights

    
    def accumulate_gradients(self, gradients_forward):
        # gradients_forward is of shape [nout]
        # self.gradients is of shape [nout, nin]
        # self.activationsin of shape [nin] -> [nout, nin]

        gradients_forward = gradients_forward * self.mask # kills gradients of dead neurons

        # multiplication of 2 1d arrays, gradients forward * activations in of shape (nout, nin)

        np.multiply(gradients_forward[:, np.newaxis], self.activationsin, out=self._tmp)
        self.gradients += self._tmp

        # bias gradient have a local derivative thats always 1 so we only chain to upstream derivative
        self.bias_gradients += gradients_forward

        # gradient at each input, equals the sum of (each weight connected to it) multiplied by its upstream gradient (forward gradient)
        
        gradients_backward = gradients_forward @ self.weights
        return gradients_backward

    
        

    def zero_grad(self):
        self.gradients.fill(0)
        self.bias_gradients.fill(0)

    # def update_weights(self, learning_rate):
    #     np.multiply(learning_rate, self.gradients, out=self._tmp) # weights
        
    #     self.weights -= self._tmp
    #     self.bias_weights -= self.bias_gradients * learning_rate

    def parameters(self):
        yield self.weights, self.gradients
        yield self.bias_weights, self.bias_gradients

    def __repr__(self):
        nin = self.weights.shape[1]
        nout = self.weights.shape[0]
        
        lines = [f"Layer(nin={nin}, nout={nout}, nonlin={self.nonlin}):"]
        
        # Format weights matrix and bias column
        for i in range(nout):
            # Format each weight with fixed total width (e.g., 7 chars, 3 decimals, space for sign)
            w_row = " ".join(f"{w: 7.3f}" for w in self.weights[i])
            bias = f"{self.bias_weights[i]: 7.3f}"
            mask = f"{int(self.mask[i])}"
            
            lines.append(f"  Neuron {i}: W = [{w_row} ]  b = {bias}  mask = {mask}")
            
        return "\n".join(lines)

    
class MLP(Model):
    def __init__(self, nin, nouts): # int, int[]
        self.layers = []

        # making all layers nonlin (relu) except the last

        if len(nouts) > 1: self.layers.append(Layer(nin=nin, nout=nouts[0], nonlin=True))
        else: self.layers.append(Layer(nin=nin, nout=nouts[0], nonlin=False))

        for i, width in enumerate(nouts[1:-1], start=1):
            self.layers.append(Layer(nin=nouts[i-1], nout=width, nonlin=True, seed=42 + i))

        if len(nouts) > 1: self.layers.append(Layer(nin=nouts[-2], nout=nouts[-1], nonlin=False))

    def __call__(self, activations): # single inference
        for layer in self.layers:
            activations = layer(activations)
        return activations

    def forward_batch(self, activations):
        for layer in self.layers:
            activations = layer.forward_batch(activations)
        return activations

    def parameters(self):
        for layer in self.layers:
            yield from layer.parameters()
            

    def accumulate_gradients(self, gradients_forward):

        for layer in reversed(self.layers):
            gradients_forward = layer.accumulate_gradients(gradients_forward=gradients_forward)

    def accumulate_gradients_batch(self, gradients_forward):
        for layer in reversed(self.layers):
            gradients_forward = layer.accumulate_gradients_batch(gradients_forward)

    def __repr__(self):
            out = ""
            for k, layer in enumerate(self.layers):
                out += f"Layer {k}: {str(layer)}\n\n"
            return out
    
    def stats(self):
        header = f"{'Layer':<7}{'Inputs':>10}{'Neurons':>10}{'Params':>12}  {'Activation':<10}"
        line = "-" * len(header)

        print(line)
        print(header)
        print(line)

        total = 0
        for k, layer in enumerate(self.layers):
            params = layer.nout * (layer.nin + 1)   # weights + biases
            total += params
            act = "ReLU" if layer.nonlin else "Linear"
            print(f"{k:<7}{layer.nin:>10,}{layer.nout:>10,}{params:>12,}  {act:<10}")

        print(line)
        print(f"Layers: {len(self.layers)}")
        print(f"Total parameters: {total:,}")
        print(f"Memory (weights, float32): {total * 4 / 1e6:.2f} MB")
        print(line)


class SGD(Optimizer): # Standard Gradient Decent, dumb decent
    def __init__(self, model: Model, learning_rate):
        self.model = model
        self.learning_rate = learning_rate

    def step_accumulate_gradients(self, x, y): # accumulates allowing for chunking

        z = self.model(x)
        loss, grad = z_loss_and_gradient(z=z, y=y)
        self.model.accumulate_gradients(gradients_forward=grad)

        return loss
    
    def step_learn(self):
        for p, g in self.model.parameters():
            g *= self.learning_rate              # in place, no temp
            p -= g   
        self.model.zero_grad()

    def step_accumulate_batch(self, x, y):  # x: (B, nin), y: (B, nout)
        z = self.model.forward_batch(x)
        loss, grad = z_loss_and_gradient_batch(z=z, y=y)
        self.model.accumulate_gradients_batch(gradients_forward=grad)
        return loss



def z_loss_and_gradient_batch(z, y):  # z, y: (B, nclasses), loss summed over the batch
    z_shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    sum_exp = exp_z.sum(axis=1, keepdims=True)
    grad = exp_z / sum_exp - y
    loss = (np.log(sum_exp) * y.sum(axis=1, keepdims=True)).sum() - (y * z_shifted).sum()
    return float(loss), grad

def z_loss_and_gradient(z, y):
    z_shifted = z - z.max()
    exp_z = np.exp(z_shifted)
    sum_exp = exp_z.sum()
    grad = exp_z / sum_exp - y
    loss = np.log(sum_exp) * y.sum() - y @ z_shifted   # -sum(y * log_softmax)
    return float(loss), grad

def z_loss(z, y): # only use when gradient is not needed, else prefer z_loss_and_gradient
    z_shifted = z - np.max(z)                      # for numerical stability
    log_probs = z_shifted - np.log(np.sum(np.exp(z_shifted)))   # log-softmax
    return -np.sum(y * log_probs)
    

In [94]:
correct = 0
test_loss = 0.0
for x, y in zip(X_test, Y_test):
    z = model(x)
    correct += z.argmax() == y.argmax()
    test_loss += z_loss(z, y)

print(f"test accuracy: {correct / len(X_test):.4f}  test loss: {test_loss / len(X_test):.4f}")

test accuracy: 0.9785  test loss: 0.0770


In [92]:
optimizer.learning_rate = 0.0002
trainer.train(n_steps=1_250, report_freq=200)

step: 0 loss = 0.02555738389492035
step: 200 loss = 0.030402718111872673
step: 400 loss = 0.03359708562493324
step: 600 loss = 0.012737827375531197
step: 800 loss = 0.020169267430901527
step: 1000 loss = 0.0
step: 1200 loss = 0.0


In [56]:
# BATCHING
model = MLP(nin=14*14, nouts=[11*11, 8*8, 5*5, 10], )

optimizer = SGD(model=model, learning_rate=0.001)

trainer = Trainer(optimizer=optimizer, x_train_set=X_train, y_train_set=Y_train, batch_size=64, batching=True)

%prun -s cumtime trainer.train(n_steps=1_250, report_freq=200)

step: 0 loss = 2.368436336517334
step: 200 loss = 0.5855403542518616
step: 400 loss = 0.2646484971046448
step: 600 loss = 0.15257644653320312
step: 800 loss = 0.19616734981536865
step: 1000 loss = 0.0
step: 1200 loss = 0.0
 

         94021 function calls (94016 primitive calls) in 0.693 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1250    0.004    0.000    0.594    0.000 3363355070.py:253(step_accumulate_batch)
        7    0.000    0.000    0.516    0.074 selectors.py:310(select)
        7    0.006    0.001    0.516    0.074 base_events.py:1976(_run_once)
     1250    0.006    0.000    0.358    0.000 3363355070.py:202(accumulate_gradients_batch)
     5000    0.327    0.000    0.353    0.000 3363355070.py:105(accumulate_gradients_batch)
     1250    0.004    0.000    0.182    0.000 3363355070.py:187(forward_batch)
     5000    0.178    0.000    0.178    0.000 3363355070.py:96(forward_batch)
      2/1    0.000    0.000    0.176    0.176 {built-in method builtins.exec}
      2/1    0.001    0.000    0.175    0.175 <string>:1(<module>)
        1    0.001    0.001    0.087    0.087 3363355070.py:14(train)
     1250    0.061    0.000    0.

In [ ]:
# CHUNKING
model = MLP(nin=14*14, nouts=[11*11, 8*8, 5*5, 10], )

optimizer = SGD(model=model, learning_rate=0.0005)

trainer = Trainer(optimizer=optimizer, x_train_set=X_train, y_train_set=Y_train, chunk_size=12)

%prun -s cumtime trainer.train(n_steps=5_000, report_freq=200)

step: 0 loss = 2.360115965207418
step: 200 loss = 2.096742401520411
step: 400 loss = 1.6216352085272472
step: 600 loss = 1.2174961119890213
step: 800 loss = 0.9954172099630038
step: 1000 loss = 1.016599090769887
step: 1200 loss = 0.7873074933886528
step: 1400 loss = 0.8926247823983431
step: 1600 loss = 0.3017973768680046
step: 1800 loss = 0.24736030467708284
step: 2000 loss = 0.5121525106951594
step: 2200 loss = 0.8812467732156316
step: 2400 loss = 0.22737260532449
step: 2600 loss = 0.879055187406872
step: 2800 loss = 0.5417098855444541
step: 3000 loss = 0.7798088946340916
step: 3200 loss = 0.3630200175878902
step: 3400 loss = 0.15239355377464867
step: 3600 loss = 0.22232822282239795
step: 3800 loss = 0.48349087113941397
step: 4000 loss = 0.43270284505949047
step: 4200 loss = 0.17496728023009686
step: 4400 loss = 0.188031110757341
step: 4600 loss = 0.14716170984077812
step: 4800 loss = 0.3927664172564012
 

         1517571 function calls (1517570 primitive calls) in 5.441 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       72    0.002    0.000    5.264    0.073 base_events.py:1976(_run_once)
     5000    0.030    0.000    5.165    0.001 {built-in method builtins.sum}
    65000    0.068    0.000    5.111    0.000 677026883.py:22(<genexpr>)
    60000    0.101    0.000    5.039    0.000 677026883.py:237(step_accumulate_gradients)
       71    0.000    0.000    5.035    0.071 events.py:92(_run)
       71    0.019    0.000    5.034    0.071 {method 'run' of '_contextvars.Context' objects}
    60000    0.150    0.000    2.779    0.000 677026883.py:195(accumulate_gradients)
   240000    2.629    0.000    2.629    0.000 677026883.py:111(accumulate_gradients)
    60000    0.147    0.000    1.405    0.000 677026883.py:180(__call__)
   240000    1.258    0.000    1.258    0.000 677026883.py:85(__call__)
    60000    0.429    0.000 